In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sp

In [2]:
tissues = ["CaH", "NAC_NACc_NACs", "Pu", "ic"]
ct_to_use = {
        "STR_D1D2_Hybrid_MSN": "eMSN_D1D2", 
        "STRd_D1_Matrix_MSN_STRv_D1_MSN": "Matrix_D1", 
        "STRd_D2_Matrix_MSN_STRv_D2_MSN": "Matrix_D2", 
        "STRd_D1_Striosome_MSN": "Patch_D1", 
        "STRd_D2_Striosome_MSN": "Patch_D2", 
}

In [6]:
import os

base_folder = os.path.expanduser("~/remote_volumes/broad/bican_um1_mccarroll/RNAseq/analysis/CAP_freeze_3_analysis/single_cell_aggregation/out")
save_folder = os.path.expanduser("~/myworkdir/XDP/data/BICAN/whole_BICAN_ezra")
metadata_path = os.path.expanduser("~/remote_volumes/broad/bican_um1_mccarroll/RNAseq/analysis/cellarium_upload/CAP_freeze_3/CAP_cell_metadata.annotated.txt.gz")

### Add metadata

In [4]:
meta_col_keep = [
    'index', 'prefix', 'cell_barcode', 'expression_doublet', 'num_genic_reads', 'num_transcripts', 'num_genes', 'num_retained_transcripts', 
    'pct_coding', 'pct_utr', 'pct_intergenic', 'pct_intronic', 'pct_mt', 'frac_contamination', 
    'donor_external_id', 
    'biobank', 'cohort', 
    'age', 'race', 'pmi_hr', 'imputed_sex', 
    #'eur_adj', 'afr_adj', 'nat_adj', 'eas_adj', 'sas_adj', 
    #'metadata_donor_outlier', 
    #'apoe_status', 'apoe_score', 
    'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 
    #'toxicology_group', 'toxicology_report_complete', 'toxicology_compounds_detected', 'hbcac_status', 'experiment', 
    #'scpred_cortex_class', 'scpred_cortex_class_max_prob', 'scpred_cortex_gaba_sub_class', 'scpred_cortex_gaba_sub_class_max_prob', 'scpred_cortex_glut_sub_class', 'scpred_cortex_glut_sub_class_max_prob', 'scpred_caudate_class', 'scpred_caudate_class_max_prob', 'scpred_caudate_spn_di', 'scpred_caudate_spn_di_max_prob', 'scpred_caudate_spn_en', 'scpred_caudate_spn_en_max_prob', 'scpred_caudate_spn_mp', 'scpred_caudate_spn_mp_max_prob', 'scpred_class', 'scpred_sub_class', 
    #'cell_type_ontology_term_id', 
    'mmc_neighborhood_name', 'mmc_neighborhood_bootstrapping_probability', 'mmc_class_name', 'mmc_class_bootstrapping_probability', 'mmc_subclass_name', 'mmc_subclass_bootstrapping_probability', 'mmc_group_name', 'mmc_group_bootstrapping_probability', 'mmc_cluster_name', 'mmc_cluster_bootstrapping_probability', 
    #'annotation_neighborhood', 'annotation_class', 'annotation_sub_class', 'annotation_notes', 'annotation_most_specific', 
    #'development_stage_ontology_term_id', 'sex_ontology_term_id', 'neuropath_diagnosis', 'disease_ontology_term_id', 
    'village', 'brain_region_abbreviation', #'tissue_ontology_term_id', 'brain_region_abbreviation_simple', 
    'ctp_donor_outlier',  # --> same as outlier file
    'ctp_sample_outlier', 'gex_donor_outlier', 'gex_donor_celltype_outlier', 
    #'n_nuclei_donor_village', 'n_nuclei_donor_village_z_score', 'n_nuclei_donor_village_exclude', 'n_umi_donor_cell_type_exclude', 'pulverized', 'single_cell_assay', 'assay_ontology_term_id', 'organism_ontology_term_id', 'is_primary_data', 'suspension_type', 'tissue_type', 'self_reported_ethnicity_ontology_term_id', 'data_freeze', 'subdissection_team'
]


rename_dict = {
    "donor_external_id": "donor_id",
    "ctp_sample_outlier": "sample_outlier",
    "ctp_donor_outlier": "donor_outlier",
    "imputed_sex": "sex"
}

In [7]:
meta_df = pd.read_csv(metadata_path, sep="\t", usecols=meta_col_keep)
meta_df = meta_df.rename(columns=rename_dict).set_index("index")
meta_df.index.name = None

meta_df

/tmp/ipykernel_112637/2515267225.py:1: DtypeWarning: Columns (80,81,82,83) have mixed types. Specify dtype option on import or set low_memory=False.
  meta_df = pd.read_csv(metadata_path, sep="\t", usecols=meta_col_keep)


,prefix,cell_barcode,expression_doublet,num_genic_reads,num_transcripts,num_genes,num_retained_transcripts,pct_coding,pct_utr,pct_intergenic,...,mmc_group_name,mmc_group_bootstrapping_probability,mmc_cluster_name,mmc_cluster_bootstrapping_probability,village,brain_region_abbreviation,donor_outlier,sample_outlier,gex_donor_outlier,gex_donor_celltype_outlier
v7_10X-GEX-3P_DFC_rxn1_TTCTAACCAATTCGTG,v7_10X-GEX-3P_DFC_rxn1,TTCTAACCAATTCGTG,False,183875,85098,9952,85066,0.0922,0.0575,0.0670,...,AMY-SLEA-BNST GABA,0.53,Human-456,0.91,v7,DFC,NaN,NaN,NaN,NaN
v7_10X-GEX-3P_DFC_rxn1_CTCCTCCGTTGGACTT,v7_10X-GEX-3P_DFC_rxn1,CTCCTCCGTTGGACTT,False,184191,84071,9887,84058,0.1023,0.0583,0.0641,...,AMY-SLEA-BNST GABA,0.43,Human-456,1.00,v7,DFC,NaN,NaN,NaN,NaN
v7_10X-GEX-3P_DFC_rxn1_AATTTCCGTGGCTTGC,v7_10X-GEX-3P_DFC_rxn1,AATTTCCGTGGCTTGC,False,167443,75596,9891,75573,0.1496,0.1210,0.0623,...,AMY-SLEA-BNST GABA,0.71,Human-456,0.98,v7,DFC,NaN,NaN,NaN,NaN
v7_10X-GEX-3P_DFC_rxn1_CTGTAGAAGCTATCCA,v7_10X-GEX-3P_DFC_rxn1,CTGTAGAAGCTATCCA,False,133341,60931,9429,60877,0.1110,0.0730,0.0655,...,AMY-SLEA-BNST GABA,0.51,Human-456,1.00,v7,DFC,NaN,NaN,NaN,NaN
v7_10X-GEX-3P_DFC_rxn1_AAGGAATCAACAGCTT,v7_10X-GEX-3P_DFC_rxn1,AAGGAATCAACAGCTT,False,137069,60870,9170,60820,0.0969,0.0614,0.0687,...,AMY-SLEA-BNST GABA,0.69,Human-456,1.00,v7,DFC,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
v94_10X-GEMX-3P_NAC_rxn8_CATCTAAGTTTCGTCC,v94_10X-GEMX-3P_NAC_rxn8,CATCTAAGTTTCGTCC,False,8795,2818,1237,2818,0.1868,0.0693,0.0825,...,Astrocyte,1.00,Human-14,0.91,v94,NAC,NaN,NaN,NaN,NaN
v94_10X-GEMX-3P_NAC_rxn8_ACATTCAGTGGGTAGT,v94_10X-GEMX-3P_NAC_rxn8,ACATTCAGTGGGTAGT,False,8212,2650,998,2650,0.2985,0.0633,0.0692,...,Oligo PLEKHG1,0.99,Human-15,0.68,v94,NAC,NaN,unexpected neurons,NaN,NaN
v94_10X-GEMX-3P_NAC_rxn8_AGTGAATGTGCAATTC,v94_10X-GEMX-3P_NAC_rxn8,AGTGAATGTGCAATTC,False,7727,2369,838,2369,0.1739,0.0751,0.0570,...,Oligo OPALIN,1.00,Human-1,1.00,v94,NAC,NaN,NaN,NaN,NaN
v94_10X-GEMX-3P_NAC_rxn8_GGAGAAGGTTGGATCT,v94_10X-GEMX-3P_NAC_rxn8,GGAGAAGGTTGGATCT,False,5619,1728,819,1728,0.2394,0.0716,0.0523,...,Oligo OPALIN,1.00,Human-1,1.00,v94,NAC,NaN,NaN,NaN,NaN


### Build one AnnData per cell type (concatenated across tissues) and attach metadata

In [7]:
# readsgene symbols
df_gene = pd.read_csv("/home/gdallagl/myworkdir/XDP/data/BioMart/ensamble-name_biomart.txt", sep="\t", header=None, names=["gene_id", "gene_symbol"])
df_gene

,gene_id,gene_symbol
0,Gene stable ID,Gene name
1,ENSG00000210049,MT-TF
2,ENSG00000211459,MT-RNR1
3,ENSG00000210077,MT-TV
4,ENSG00000210082,MT-RNR2
...,...,...
86365,ENSG00000168710,AHCYL1
86366,ENSG00000081692,JMJD4
86367,ENSG00000157873,TNFRSF14
86368,ENSG00000132676,DAP3


In [8]:
import scipy.io as sio

for ct, ct_for_deg in list(ct_to_use.items())[1:2]:

    print(f"Processing cell type: {ct} -> {ct_for_deg}")

    tissue_adatas = {}
    for tissue in tissues:

        print(f"\tProcessing tissue: {tissue}")

        folder = f"{base_folder}/{ct}__{tissue}"

        barcodes = pd.read_csv(f"{folder}/barcodes.tsv.gz", header=None, sep="\t")[0].tolist()
        features = pd.read_csv(f"{folder}/features.tsv.gz", header=None, sep="\t")[0].tolist()

        # matrix.mtx is already cells x genes (rows=barcodes, cols=features) -> no transpose needed
        counts = sio.mmread(f"{folder}/matrix.mtx.gz").tocsr()

        tissue_adatas[tissue] = sc.AnnData(
            X=counts,
            obs=pd.DataFrame(index=barcodes),
            var=pd.DataFrame(index=features),
        )

    # one AnnData per cell type, concatenated across all tissues
    adata = sc.concat(tissue_adatas, label="tissue", join="outer")
    adata.obs["ct_for_deg"] = ct_for_deg

    # attach the selected per-cell metadata columns, matched by barcode
    adata.obs = adata.obs.join(meta_df, how="left")

    # layers
    adata.layers["counts"] = adata.X.copy()  # make sure raw counts are in .layers["counts"]
    #adata.X = sp.csr_matrix(adata.shape, dtype=np.float32)  # all zeros

    # give cgene symbol names
    adata.var["gene_symbol"] = adata.var.join(df_gene.set_index("gene_id"), how="left")["gene_symbol"].fillna(adata.var_names.to_series())
    adata.var["ensembl_id"] = adata.var_names
    adata.var_names = adata.var["gene_symbol"].astype(str).values
    adata.var_names_make_unique()


    # some columns mix numeric and string values -> object dtype with
    for col in adata.obs.select_dtypes(include="object").columns:
        adata.obs[col] = adata.obs[col].apply(lambda x: x if pd.isna(x) else str(x))

    os.makedirs(f"{save_folder}/{ct_for_deg}", exist_ok=True)
    adata.write_h5ad(f"{save_folder}/{ct_for_deg}/{ct_for_deg}.h5ad")
    print(ct_for_deg, adata)

    break

Processing cell type: STRd_D1_Matrix_MSN_STRv_D1_MSN -> Matrix_D1
	Processing tissue: CaH
	Processing tissue: NAC_NACc_NACs
	Processing tissue: Pu
	Processing tissue: ic
Matrix_D1 AnnData object with n_obs × n_vars = 441953 × 38095
    obs: 'tissue', 'ct_for_deg', 'prefix', 'cell_barcode', 'expression_doublet', 'num_genic_reads', 'num_transcripts', 'num_genes', 'num_retained_transcripts', 'pct_coding', 'pct_utr', 'pct_intergenic', 'pct_intronic', 'pct_mt', 'frac_contamination', 'donor_id', 'biobank', 'cohort', 'age', 'race', 'pmi_hr', 'sex', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'mmc_neighborhood_name', 'mmc_neighborhood_bootstrapping_probability', 'mmc_class_name', 'mmc_class_bootstrapping_probability', 'mmc_subclass_name', 'mmc_subclass_bootstrapping_probability', 'mmc_group_name', 'mmc_group_bootstrapping_probability', 'mmc_cluster_name', 'mmc_cluster_bootstrapping_probability', 'village', 'brain_region_abbreviation', 'donor_outlier', 'sample_outlier', 'gex_donor_outlier', 'gex_donor_c

In [11]:
tmp = meta_df.drop_duplicates(subset="donor_id")
tmp["age"].describe()


count    178.000000
mean      61.488764
std       18.367633
min       27.000000
25%       47.250000
50%       62.000000
75%       75.750000
max       90.000000
Name: age, dtype: float64